# AML s26 - Regression with Qualitative and Quantitative Features


## 1. Import Required Libraries

In [1]:
# Import necessary libraries for data manipulation, visualization, and machine learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## 2. Load and Explore the Dataset

In [2]:
df = pd.read_csv('house_data.csv')

### 2.1 Dataset Overview

In [3]:
# Display basic information about the dataset
print("-" * 20)
print("Dataset overview")

# Show the dimensions of the dataset (rows, columns)
print(f"\nDataset shape: {df.shape}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")

--------------------
Dataset overview

Dataset shape: (500, 8)
Number of samples: 500
Number of features: 8


In [4]:
# Display the first 10 rows of the dataset to understand its structure
print("\nFirst few rows:")
df.head(10)


First few rows:


,square_footage,bedrooms,age_years,distance_to_city,neighborhood,house_style,condition,price
0,1811.258321,1,16.793293,13.944691,Waterfront,Victorian,Good,541117.540235
1,3366.928627,1,41.255367,26.062269,Rural,Victorian,Good,504465.969652
2,2776.383643,1,18.152961,17.186067,Waterfront,Colonial,Excellent,686803.135889
3,2416.377907,5,1.711406,16.657269,Rural,Colonial,Good,476844.265578
4,1221.250329,4,41.532725,1.991445,Rural,Victorian,Poor,201633.437586
5,1221.185205,5,17.259607,29.614096,Rural,Ranch,Good,309216.513114
6,956.825753,4,38.691726,4.566153,Suburbs,Victorian,Poor,250175.208672
7,3138.675594,5,18.137895,7.702796,Waterfront,Modern,Good,848322.606245
8,2423.010532,5,43.053401,2.499208,Rural,Modern,Excellent,529228.961667
9,2711.795960,3,10.975526,19.560086,Downtown,Colonial,Excellent,583432.321631


In [5]:
# Check the data types of each column

print("\nData types:")
print(df.dtypes)


Data types:
square_footage      float64
bedrooms              int64
age_years           float64
distance_to_city    float64
neighborhood            str
house_style             str
condition               str
price               float64
dtype: object


### 2.2 Statistical Summary
Generate descriptive statistics for numerical features. This includes count, mean, std, min, max, and quartiles.

In [6]:
print("\nBasic statistics:")
df.describe()


Basic statistics:


,square_footage,bedrooms,age_years,distance_to_city,price
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,2146.116623,2.958000,25.401785,15.286203,468879.598292
std,806.458703,1.440968,14.471390,8.210046,143551.193315
min,813.666276,1.000000,0.011876,1.009643,101284.352124
25%,1451.455165,2.000000,13.141060,8.028797,363111.559021
50%,2185.542121,3.000000,25.960712,15.605278,461810.689911
75%,2841.537180,4.000000,37.721654,22.514261,573256.298659
max,3481.004950,5.000000,49.967675,29.868934,848322.606245


### 2.3 Categorical Feature Distributions

Examine the distribution of categorical features.
This helps understand the balance of categories and potential class imbalances.

In [7]:
print("\nCategorical feature distributions:")

print("\n--- Neighborhood: ---")
print(df['neighborhood'].value_counts())

print("\n--- House Style: ---")
print(df['house_style'].value_counts())

print("\n--- Condition: ---")
print(df['condition'].value_counts())


Categorical feature distributions:

--- Neighborhood: ---
neighborhood
Waterfront    141
Suburbs       129
Downtown      117
Rural         113
Name: count, dtype: int64

--- House Style: ---
house_style
Ranch        135
Victorian    132
Modern       125
Colonial     108
Name: count, dtype: int64

--- Condition: ---
condition
Good         217
Fair         157
Excellent     71
Poor          55
Name: count, dtype: int64


## 3. Prepare Features and Target Variable

Separate the features (X) from the target variable (y) 

In [8]:
X = df.drop('price', axis=1)
y = df['price']

In [9]:
# Define which features are categorical and which are numerical
# This is important for proper preprocessing
categorical_features = ['neighborhood', 'house_style', 'condition']
numerical_features = ['square_footage', 'bedrooms', 'age_years', 'distance_to_city']

print(f"\n\nQuantitative features: {numerical_features}")
print(f"Qualitative features: {categorical_features}")



Quantitative features: ['square_footage', 'bedrooms', 'age_years', 'distance_to_city']
Qualitative features: ['neighborhood', 'house_style', 'condition']


## 4. Split Data into Training and Testing Sets

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")


Training set size: 400 samples
Test set size: 100 samples


## 5. Create Preprocessing Pipeline

We need to preprocess our features differently based on their type:
- **Numerical features**: Standardized (scaled to mean=0, std=1)
- **Categorical features**: One-hot encoded (converted to binary dummy variables)

In [11]:
# Create a ColumnTransformer to apply different preprocessing to different feature types
# - StandardScaler: Standardizes numerical features (subtracts mean, divides by std)
# - OneHotEncoder: Converts categorical variables into binary columns
#   - drop='first': Drops the first category to avoid multicollinearity (dummy variable trap)
#   - sparse_output=False: Returns a dense array instead of sparse matrix
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

## 6. Build and Train the Model

Create a Linear Regression model.

In [12]:
model = LinearRegression()


Create a pipeline that chains preprocessing and model training, ensuring that:
- Training data is preprocessed before training
- Test data is preprocessed using the same transformations (fitted on training data)

Train the model on the training data.

In [13]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
])

pipeline.fit(X_train, y_train)
print("Model training complete!")

Model training complete!


## 7. Make Predictions
- Generate predictions on both training and testing sets
- Training predictions: to check for overfitting
- Testing predictions: to evaluate model performance on unseen data

In [14]:

y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

## 8. Evaluate Model Performance

We use multiple metrics to assess model quality:
- **R² (R-squared)**: Proportion of variance explained (0-1, higher is better)
- **RMSE (Root Mean Squared Error)**: Average prediction error in original units
- **MAE (Mean Absolute Error)**: Average absolute prediction error

In [15]:
# Calculate performance metrics

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"Training R²: {train_r2:.4f}")
print(f"Training RMSE: ${train_rmse:,.2f}")
print('-'*20)
print(f"Test R²: {test_r2:.4f}")
print(f"Test RMSE: ${test_rmse:,.2f}")
print(f"Test MAE: ${test_mae:,.2f}")


Training R²: 0.9545
Training RMSE: $31,386.08
--------------------
Test R²: 0.9284
Test RMSE: $34,018.97
Test MAE: $26,684.51


## 9. Interpret Model Coefficients

The coefficients tell us how much each feature affects the predicted price.
- **Positive coefficient**: Feature increases price
- **Negative coefficient**: Feature decreases price
- **Magnitude**: How strong the effect is

In [16]:
print("\n" + "-" * 20)
print("Model Coefficients")

# Get the names of all features after preprocessing
# Numerical features keep their names
# Categorical features are expanded into multiple binary columns
feature_names = ( numerical_features + list(pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)) )

# Extract the trained linear regression model from the pipeline
lr_model = pipeline.named_steps['regressor']

# Get the coefficients (weights) learned by the model
coefficients = lr_model.coef_

# Create a DataFrame to display coefficients in a readable format
# Sort by coefficient value to see most positive and negative effects
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

print("\nModel Coefficients (sorted by impact):")
print(coef_df.to_string(index=False))

# Display the intercept (baseline price when all features = 0)
print(f"\nIntercept (baseline): ${lr_model.intercept_:,.2f}")


--------------------
Model Coefficients

Model Coefficients (sorted by impact):
                feature    coefficient
         square_footage  120932.261323
neighborhood_Waterfront   68948.968669
     house_style_Modern   46559.296594
  house_style_Victorian   29530.158701
               bedrooms   20912.026618
      house_style_Ranch    3778.641141
       distance_to_city  -22493.153821
              age_years  -27756.369757
   neighborhood_Suburbs  -41153.024336
         condition_Good  -41489.764302
         condition_Fair  -80281.654498
     neighborhood_Rural  -80872.954769
         condition_Poor -114367.677291

Intercept (baseline): $515,057.60


## 10. Make Predictions on New Data

Let's test the model with two example houses to see how different neighborhoods affect price.

### Example 1: Waterfront House

In [17]:
print("\n" + "-" * 20)
print("EXAMPLE PREDICTION")

# Create a DataFrame with features for a new house
# Note: The DataFrame must have the same column names and order as training data
new_house = pd.DataFrame({
    'square_footage': [2500],
    'bedrooms': [4],
    'age_years': [10],
    'distance_to_city': [8],
    'neighborhood': ['Waterfront'],  # Premium location
    'house_style': ['Modern'],
    'condition': ['Excellent']
})

print("\nNew house features - 'Waterfront':")
print(new_house.to_string(index=False))

# Use the trained pipeline to make a prediction
# The pipeline automatically applies the same preprocessing as during training
predicted_price = pipeline.predict(new_house)[0]
print(f"\nPredicted price: ${predicted_price:,.2f}")


--------------------
EXAMPLE PREDICTION

New house features - 'Waterfront':
 square_footage  bedrooms  age_years  distance_to_city neighborhood house_style condition
           2500         4         10                 8   Waterfront      Modern Excellent

Predicted price: $745,272.98


### Example 2: Rural House

In [18]:
# Same house but in a different neighborhood
# This allows us to isolate the effect of neighborhood on price
new_house_rural = pd.DataFrame({
    'square_footage': [2500],
    'bedrooms': [4],
    'age_years': [10],
    'distance_to_city': [8],
    'neighborhood': ['Rural'],  # Different location
    'house_style': ['Modern'],
    'condition': ['Excellent']
})

print("\nNew house features - 'Rural':")
print(new_house_rural.to_string(index=False))

# Predict price for the rural house
predicted_price_rural = pipeline.predict(new_house_rural)[0]
print(f"\nPredicted price: ${predicted_price_rural:,.2f}")

# Calculate the price difference due to neighborhood
price_difference = predicted_price - predicted_price_rural
print(f"\nPrice difference (Waterfront vs Rural): ${price_difference:,.2f}")
print(f"Percentage difference: {(price_difference/predicted_price_rural)*100:.1f}%")


New house features - 'Rural':
 square_footage  bedrooms  age_years  distance_to_city neighborhood house_style condition
           2500         4         10                 8        Rural      Modern Excellent

Predicted price: $595,451.05

Price difference (Waterfront vs Rural): $149,821.92
Percentage difference: 25.2%
